# 03 — Feature engineering

**Cilj:** napraviti jedan reproducibilan train/test split i uporediva raw, naive per-90 i reliability-adjusted obeležja bez curenja informacija iz test skupa.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.features import (
    AVAILABILITY_FEATURES,
    CATEGORICAL_FEATURES,
    PER90_FEATURES,
    PERCENTAGE_FEATURES,
    RAW_NUMERIC_FEATURES,
    add_previous_market_value,
    build_feature_sets,
)

final_df = pd.read_csv(PROCESSED_DIR / "final_players.csv")
valuations = pd.read_csv(RAW_DIR / "player_valuations.csv")
final_df = add_previous_market_value(final_df, valuations)
print(f"Players: {len(final_df):,}")

## Target i missing indikatori

Modeli koriste `log_market_value = log1p(market_value_in_eur)`. Za `Cmp%`, `Succ%` i `Won%` čuvamo i binarni indikator dostupnosti: missing procenat često znači da igrač nije imao nijedan pokušaj, što je informacija, a ne samo problem za imputaciju.

In [ ]:
train_df, test_df = build_feature_sets(final_df, test_size=0.20, random_state=42, shrinkage_k=450)

print(f"Train: {len(train_df):,}; test: {len(test_df):,}")
display(train_df[["market_value_in_eur", "log_market_value", *AVAILABILITY_FEATURES]].head())

## Zašto naive per-90 može da eksplodira

Za counting statistiku `x`, naive stopa je `90x / Min`. Jedan događaj u 10 minuta postaje 9 događaja per 90, iako je uzorak veoma slab. Zato stopu skupljamo ka pozicionom proseku:

\[
w=\frac{Min}{Min+450}, \qquad
x_{adj}=w\,x_{per90}+(1-w)\,\bar{x}_{position}
\]

Sa malo minuta dominira baseline; sa mnogo minuta dominira lična stopa. Pozicioni baseline se uči **isključivo na train skupu**.

In [ ]:
example_columns = ["Player", "Pos", "Min", "Tkl_per90", "Tkl_per90_adjusted"]
display(train_df[example_columns].sort_values("Tkl_per90", ascending=False).head(15))

weights = pd.Series([10, 90, 450, 900, 1800], name="Min").to_frame()
weights["w = Min / (Min + 450)"] = weights["Min"] / (weights["Min"] + 450)
weights

## Feature porodice

In [ ]:
feature_sets = {
    "raw_numeric": [*RAW_NUMERIC_FEATURES, *AVAILABILITY_FEATURES],
    "per90_numeric": [
        "Age", "Min", *[f"{f}_per90" for f in PER90_FEATURES],
        *PERCENTAGE_FEATURES, *AVAILABILITY_FEATURES,
    ],
    "shrunk_numeric": [
        "Age", "Min", *[f"{f}_per90_adjusted" for f in PER90_FEATURES],
        *PERCENTAGE_FEATURES, *AVAILABILITY_FEATURES,
    ],
    "categorical": CATEGORICAL_FEATURES,
}
pd.Series({name: len(columns) for name, columns in feature_sets.items()}, name="columns")

## Snimanje train/test skupova

Indeksi se čuvaju da svi modeli A–I koriste potpuno isti split.

In [ ]:
train_df.to_csv(PROCESSED_DIR / "train_features.csv", index=True, index_label="split_index")
test_df.to_csv(PROCESSED_DIR / "test_features.csv", index=True, index_label="split_index")

assert set(train_df.index).isdisjoint(test_df.index)
assert len(train_df) + len(test_df) == len(final_df)
print(f"Saved feature sets to {PROCESSED_DIR}")